In [31]:
!pip install nltk gensim scikit-learn tqdm

In [32]:
# Βασικές βιβλιοθήκες
import os
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ML
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [33]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
#Φορτώνουμε τα δεδομένα
def load_data(path):
    texts, labels = [], []
    for label in ['pos', 'neg']:
        folder = os.path.join(path, label)
        for file in os.listdir(folder):
            with open(os.path.join(folder, file), encoding='latin-1') as f:
                texts.append(f.read())
                labels.append(1 if label == 'pos' else 0)
    return np.array(texts), np.array(labels)

X, y = load_data('/content/drive/MyDrive/datasets/txt_sentoken')

df = pd.DataFrame({'text': X, 'label': y})

print("Total:", len(X))
print("Positive:", sum(y), "Negative:", len(y)-sum(y))


Total: 2000
Positive: 1000 Negative: 1000


In [36]:
#1o Baseline-Tokenization +Lowercasing+ Stopword removal

stop_words = set(stopwords.words('english'))

def baseline_preprocess(text):
    # lowercase
    text = text.lower()

    # tokenization
    tokens = nltk.word_tokenize(text)

    # κρατάμε μόνο αλφαβητικά tokens και αφαιρούμε stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    return " ".join(tokens)


In [37]:
import nltk
nltk.download('punkt_tab')
df['clean_text'] = df['text'].apply(baseline_preprocess)
df[['text', 'clean_text']].head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,text,clean_text
0,films adapted from comic books have had plenty...,films adapted comic books plenty success wheth...
1,every now and then a movie comes along from a ...,every movie comes along suspect studio every i...
2,you've got mail works alot better than it dese...,got mail works alot better deserves order make...
3,""" jaws "" is a rare film that grabs your atten...",jaws rare film grabs attention shows single im...
4,moviemaking is a lot like being the general ma...,moviemaking lot like general manager nfl team ...


In [38]:
#2o Baseline-Tokenization +Lowercasing+ Stopword removal+Stemming
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def baseline_preprocess_stemming(text):
    # lowercase
    text = text.lower()

    # tokenization
    tokens = nltk.word_tokenize(text)

    # φιλτράρισμα + stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    # stemming
    tokens = [stemmer.stem(t) for t in tokens]

    return " ".join(tokens)


In [39]:
df['clean_text_stem'] = df['text'].apply(baseline_preprocess_stemming)
df[['clean_text', 'clean_text_stem']].head()

,clean_text,clean_text_stem
0,films adapted comic books plenty success wheth...,film adapt comic book plenti success whether s...
1,every movie comes along suspect studio every i...,everi movi come along suspect studio everi ind...
2,got mail works alot better deserves order make...,got mail work alot better deserv order make fi...
3,jaws rare film grabs attention shows single im...,jaw rare film grab attent show singl imag scre...
4,moviemaking lot like general manager nfl team ...,moviemak lot like gener manag nfl team cap era...


In [40]:
#Φτιάχνουμε 2 datasets χωρις χρήση df
X_base = [baseline_preprocess(text) for text in X]
X_stem = [baseline_preprocess_stemming(text) for text in X]


In [41]:
#Φτιάχνουμε συνάρτηση που δέχεται vectonizer και model
def evaluate_model(X, y, vectorizer, model):
    X = np.array(X)
    y = np.array(y)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)

    accs, precs, recs, f1s, cms = [], [], [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train = vectorizer.fit_transform(X_train_raw)
        X_test = vectorizer.transform(X_test_raw)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        p, r, f, _ = precision_recall_fscore_support(
            y_test, y_pred, average='macro', zero_division=0
        )
        precs.append(p); recs.append(r); f1s.append(f)
        cms.append(confusion_matrix(y_test, y_pred))

    return {
        "accuracy": np.mean(accs),
        "precision": np.mean(precs),
        "recall": np.mean(recs),
        "f1": np.mean(f1s),
        "confusion_matrix": np.sum(cms, axis=0)
    }


In [42]:
#Υλοποίηση των 4 πρώτων pipelines (Το Word2Vec παρακάτω)
pipelines = {
    "MNB_TF": (
        CountVectorizer(binary=False),
        MultinomialNB()
    ),
    "MNB_TFIDF": (
        TfidfVectorizer(max_features=6000),
        MultinomialNB()
    ),
    "BNB_Binary": (
        CountVectorizer(binary=True),
        BernoulliNB()
    ),
    "Boolean_MNB": (
        CountVectorizer(binary=True),
        MultinomialNB()
    )
}


In [43]:
#Εκτέλεση και σύγκριση baselines E1-E3
results = {}

for name, (vec, clf) in pipelines.items():
    res_base = evaluate_model(X_base, y, vec, clf)
    res_stem = evaluate_model(X_stem, y, vec, clf)

    best = res_base if res_base['f1'] >= res_stem['f1'] else res_stem

    results[name] = {
        "baseline": "no_stemming" if best == res_base else "stemming",
        "metrics": best
    }

    print(f"\n{name}")
    print("Best baseline:", results[name]["baseline"])
    print(f"Accuracy:  {best['accuracy']:.4f}")
    print(f"Precision: {best['precision']:.4f}")
    print(f"Recall:    {best['recall']:.4f}")
    print(f"F1:        {best['f1']:.4f}")
    print("Confusion matrix:\n", best["confusion_matrix"])


MNB_TF
Best baseline: stemming
Accuracy:  0.8060
Precision: 0.8064
Recall:    0.8060
F1:        0.8059
Confusion matrix:
 [[816 184]
 [204 796]]

MNB_TFIDF
Best baseline: no_stemming
Accuracy:  0.8105
Precision: 0.8129
Recall:    0.8105
F1:        0.8101
Confusion matrix:
 [[852 148]
 [231 769]]

BNB_Binary
Best baseline: no_stemming
Accuracy:  0.7880
Precision: 0.8021
Recall:    0.7880
F1:        0.7854
Confusion matrix:
 [[896 104]
 [320 680]]

Boolean_MNB
Best baseline: stemming
Accuracy:  0.8225
Precision: 0.8229
Recall:    0.8225
F1:        0.8225
Confusion matrix:
 [[833 167]
 [188 812]]


In [44]:
#Παίρνουμε τα 2 baselines
X_tokens_base = [text.split() for text in X_base]
X_tokens_stem = [text.split() for text in X_stem]

In [45]:
#Συνάρτηση mean pooling
def document_mean_vector(tokens, w2v_model, vector_size):
    vectors = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

In [46]:
from gensim.models import Word2Vec

#Evaluation function για Word2Vec (ΧΩΡΙΣ leakage)
def evaluate_word2vec(X_tokens, y, vector_size=100):
    X_tokens = np.array(X_tokens, dtype=object)
    y = np.array(y)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)

    accs, precs, recs, f1s, cms = [], [], [], [], []

    for train_idx, test_idx in skf.split(X_tokens, y):
        X_train_tokens = X_tokens[train_idx]
        X_test_tokens = X_tokens[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #  Train Word2Vec μόνο στο training fold
        w2v = Word2Vec(
            sentences=X_train_tokens,
            vector_size=vector_size,
            window=5,
            min_count=2,
            workers=4
        )

        # Mean pooling
        X_train_vec = np.array([
            document_mean_vector(doc, w2v, vector_size)
            for doc in X_train_tokens
        ])
        X_test_vec = np.array([
            document_mean_vector(doc, w2v, vector_size)
            for doc in X_test_tokens
        ])

        # MinMaxScaler (fit μόνο στο train)
        scaler = MinMaxScaler()
        X_train_vec = scaler.fit_transform(X_train_vec)
        X_test_vec = scaler.transform(X_test_vec)

        # Multinomial NB
        clf = MultinomialNB()
        clf.fit(X_train_vec, y_train)
        y_pred = clf.predict(X_test_vec)

        # Μετρικές
        accs.append(accuracy_score(y_test, y_pred))
        p, r, f, _ = precision_recall_fscore_support(
            y_test, y_pred, average='macro', zero_division=0
        )
        precs.append(p); recs.append(r); f1s.append(f)
        cms.append(confusion_matrix(y_test, y_pred))

    return {
        "accuracy": np.mean(accs),
        "precision": np.mean(precs),
        "recall": np.mean(recs),
        "f1": np.mean(f1s),
        "confusion_matrix": np.sum(cms, axis=0)
    }


In [47]:
from sklearn.preprocessing import MinMaxScaler
#Συγκριση των 2 baselines για το Ε4
res_w2v_base = evaluate_word2vec(X_tokens_base, y)
res_w2v_stem = evaluate_word2vec(X_tokens_stem, y)

if res_w2v_base['f1'] >= res_w2v_stem['f1']:
    best_w2v = res_w2v_base
    best_baseline = "no_stemming"
else:
    best_w2v = res_w2v_stem
    best_baseline = "stemming"

print("\nE4: Word2Vec + Mean Pooling + MultinomialNB")
print("Best baseline:", best_baseline)
print(f"Accuracy:  {best_w2v['accuracy']:.4f}")
print(f"Precision: {best_w2v['precision']:.4f}")
print(f"Recall:    {best_w2v['recall']:.4f}")
print(f"F1:        {best_w2v['f1']:.4f}")
print("Confusion matrix:\n", best_w2v["confusion_matrix"])


E4: Word2Vec + Mean Pooling + MultinomialNB
Best baseline: no_stemming
Accuracy:  0.5745
Precision: 0.5748
Recall:    0.5745
F1:        0.5742
Confusion matrix:
 [[587 413]
 [438 562]]


In [48]:
#Μέρος Β
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [49]:
#Διαχείριση Άρνησης
negation_words = {
    "no", "not", "never", "n't",
    "dont", "don't",
    "didnt", "didn't",
    "cant", "can't",
    "wont", "won't"
}

punctuation = {".", "!", "?", ",", ";", ":"}

def apply_negation(tokens):
    negated = []
    negate = False

    for token in tokens:
        if token in punctuation:
            negate = False
            continue

        if token in negation_words or token.endswith("n't"):
            negate = True
            continue

        if negate:
            negated.append("NOT_" + token)
        else:
            negated.append(token)

    return negated


In [50]:
#Baseline χωρίς stemming
def preprocess_B_baseline_no_stem(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(tokens)


In [51]:
#Baseline + Stemming
def preprocess_B_baseline_stem(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)


In [52]:
# Με άρνηση
#Baseline +  no Stemming + Negation
def preprocess_B_neg_no_stem(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = apply_negation(tokens)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(tokens)


#Baseline + Stemming + Negation
def preprocess_B_neg_stem(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = apply_negation(tokens)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

In [53]:
#POS weighting (ΜΕΤΑ το vectorizer)
POS_BOOST = {"JJ", "JJR", "JJS", "RB", "RBR", "RBS"}

def apply_pos_weighting(X, texts, vectorizer):
    """
    Εφαρμόζουμε βάρος 2 σε adjectives και adverbs
    μόνο για μη δυαδικές αναπαραστάσεις
    """
    X = X.tolil()
    vocab = vectorizer.vocabulary_

    for i, text in enumerate(texts):
        tokens = text.split()
        pos_tags = nltk.pos_tag(tokens)

        for token, tag in pos_tags:
            if tag in POS_BOOST and token in vocab:
                idx = vocab[token]
                X[i, idx] *= 2.0

    return X.tocsr()

In [54]:
#Sentence position weighting (ΜΕΤΑ το vectorizer)
def apply_sentence_weighting(X, raw_texts, vectorizer, preprocess_func):
    """
    Βάρος 2 για tokens που εμφανίζονται
    στην 1η ή τελευταία πρόταση
    """
    X = X.tolil()
    vocab = vectorizer.vocabulary_

    for i, raw_text in enumerate(raw_texts):
        sentences = nltk.sent_tokenize(raw_text)
        if len(sentences) == 0:
            continue

        first_tokens = preprocess_func(sentences[0]).split()
        last_tokens = preprocess_func(sentences[-1]).split()

        important_tokens = set(first_tokens + last_tokens)

        for token in important_tokens:
            if token in vocab:
                idx = vocab[token]
                X[i, idx] *= 2.0

    return X.tocsr()

In [55]:
#Evaluate συνάρτηση για το μέρος Β
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def evaluate_model_B(
    X_raw_texts, y,
    vectorizer, model,
    preprocess_func,
    use_pos=False,
    use_sentence=False
):

    X_raw_texts = np.array(X_raw_texts)
    y = np.array(y)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)

    accs, precs, recs, f1s, cms = [], [], [], [], []

    for train_idx, test_idx in skf.split(X_raw_texts, y):
        X_train_raw = X_raw_texts[train_idx]
        X_test_raw = X_raw_texts[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # preprocessing
        X_train = [preprocess_func(t) for t in X_train_raw]
        X_test = [preprocess_func(t) for t in X_test_raw]

        # vectorization
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        # POS weighting (ΟΧΙ για binary)
        if use_pos and not vectorizer.binary:
            X_train_vec = apply_pos_weighting(X_train_vec, X_train, vectorizer)
            X_test_vec = apply_pos_weighting(X_test_vec, X_test, vectorizer)

        # Sentence weighting (ΟΧΙ για binary)
        if use_sentence and not vectorizer.binary:
            X_train_vec = apply_sentence_weighting(
                X_train_vec, X_train_raw, vectorizer, preprocess_func
            )
            X_test_vec = apply_sentence_weighting(
                X_test_vec, X_test_raw, vectorizer, preprocess_func
            )

        # training
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_test_vec)

        accs.append(accuracy_score(y_test, y_pred))
        p, r, f, _ = precision_recall_fscore_support(
            y_test, y_pred, average='macro', zero_division=0
        )
        precs.append(p)
        recs.append(r)
        f1s.append(f)
        cms.append(confusion_matrix(y_test, y_pred))

    return {
        "accuracy": np.mean(accs),
        "precision": np.mean(precs),
        "recall": np.mean(recs),
        "f1": np.mean(f1s),
        "confusion_matrix": np.sum(cms, axis=0)
    }

In [56]:
pipelines_B = {
    "MNB_TF": (
        CountVectorizer(binary=False),
        MultinomialNB(),
        preprocess_B_baseline_stem
    ),
    "MNB_TFIDF": (
        TfidfVectorizer(max_features=6000),
        MultinomialNB(),
        preprocess_B_baseline_no_stem
    ),
    "BNB_Binary": (
        CountVectorizer(binary=True),
        BernoulliNB(),
        preprocess_B_baseline_no_stem
    ),
    "Boolean_MNB": (
        CountVectorizer(binary=True),
        MultinomialNB(),
        preprocess_B_baseline_stem
    )}

In [57]:
#5 ΣΕΝΆΡΙΑ
scenarios = {
    "Baseline": dict(use_pos=False, use_sentence=False),
    "Baseline + Negation": dict(use_pos=False, use_sentence=False),
    "Baseline + POS": dict(use_pos=True, use_sentence=False),
    "Baseline + Sentence": dict(use_pos=False, use_sentence=True),
    "Baseline + All": dict(use_pos=True, use_sentence=True)
}


In [58]:
nltk.download('averaged_perceptron_tagger_eng')

results_B = {}

for name, (vectorizer, model, _) in pipelines_B.items():
    print(f"\n===== {name} =====")
    results_B[name] = {}

    for scen_name, flags in scenarios.items():

        # επιλογή preprocess
        if name == "BNB_Binary":
            baseline_prep = preprocess_B_baseline_stem
            neg_prep = preprocess_B_neg_stem
        else:
            baseline_prep = preprocess_B_baseline_no_stem
            neg_prep = preprocess_B_neg_no_stem

        if scen_name in ["Baseline + Negation", "Baseline + All"]:
            preprocess_func = neg_prep
        else:
            preprocess_func = baseline_prep

        res = evaluate_model_B(
            X_raw_texts=X,
            y=y,
            vectorizer=vectorizer,
            model=model,
            preprocess_func=preprocess_func,
            use_pos=flags["use_pos"],
            use_sentence=flags["use_sentence"]
        )

        results_B[name][scen_name] = res

        print(f"\nScenario: {scen_name}")
        print(f"Accuracy:  {res['accuracy']:.4f}")
        print(f"Precision: {res['precision']:.4f}")
        print(f"Recall:    {res['recall']:.4f}")
        print(f"F1:        {res['f1']:.4f}")
        print("Confusion matrix:\n", res["confusion_matrix"])

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.



===== MNB_TF =====

Scenario: Baseline
Accuracy:  0.8025
Precision: 0.8030
Recall:    0.8025
F1:        0.8024
Confusion matrix:
 [[812 188]
 [207 793]]

Scenario: Baseline + Negation
Accuracy:  0.8040
Precision: 0.8043
Recall:    0.8040
F1:        0.8040
Confusion matrix:
 [[810 190]
 [202 798]]

Scenario: Baseline + POS
Accuracy:  0.5015
Precision: 0.4504
Recall:    0.5015
F1:        0.3366
Confusion matrix:
 [[201 799]
 [198 802]]

Scenario: Baseline + Sentence
Accuracy:  0.7970
Precision: 0.7979
Recall:    0.7970
F1:        0.7969
Confusion matrix:
 [[810 190]
 [216 784]]

Scenario: Baseline + All
Accuracy:  0.5020
Precision: 0.5505
Recall:    0.5020
F1:        0.3377
Confusion matrix:
 [[202 798]
 [198 802]]

===== MNB_TFIDF =====

Scenario: Baseline
Accuracy:  0.8105
Precision: 0.8129
Recall:    0.8105
F1:        0.8101
Confusion matrix:
 [[852 148]
 [231 769]]

Scenario: Baseline + Negation
Accuracy:  0.8055
Precision: 0.8073
Recall:    0.8055
F1:        0.8052
Confusion matrix

In [59]:
# Preprocessing για Word2Vec + Negation
def preprocess_w2v_negation(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)

    # διαχείριση άρνησης
    tokens = apply_negation(tokens)

    # κρατάμε μόνο alphabetic & αφαιρούμε stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    return tokens  # ΠΡΟΣΟΧΗ: επιστρέφει tokens, όχι string


In [60]:
# Word2Vec tokens με Διαχείριση Άρνησης (Μέρος Β)
X_tokens_w2v_neg = [preprocess_w2v_negation(text) for text in X]

In [61]:
# Μέρος Β – Word2Vec + Negation
# Σημείωση:Στο Word2Vec εφαρμόζουμε μόνο την διαχείριση άρνησης.
# Οι τεχνικές POS weighting και sentence position weighting δεν χρησιμοποιούνται, γιατί βασίζονται σε βάρη ανά token
# σε bag-of-words αναπαραστάσεις. Στο Word2Vec με mean pooling,τα embeddings των λέξεων συνδυάζονται σε ένα ενιαίο διάνυσμα
# για κάθε έγγραφο, οπότε η πληροφορία σε επίπεδο token χάνεται και οι συγκεκριμένες τεχνικές δεν μπορούν να εφαρμοστούν
# με ουσιαστικό τρόπο.


res_w2v_neg = evaluate_word2vec(X_tokens_w2v_neg, y)

print("\nE4 (Part B): Word2Vec + Mean Pooling + MNB + Negation")
print("Accuracy:", res_w2v_neg["accuracy"])
print("Precision:", res_w2v_neg["precision"])
print("Recall:", res_w2v_neg["recall"])
print("F1:", res_w2v_neg["f1"])
print("Confusion matrix:\n", res_w2v_neg["confusion_matrix"])


E4 (Part B): Word2Vec + Mean Pooling + MNB + Negation
Accuracy: 0.563
Precision: 0.5632011187393929
Recall: 0.563
F1: 0.562643743645745
Confusion matrix:
 [[575 425]
 [449 551]]
